In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: c:\Users\fabih\Desktop\CREATOR\data\data_e-commerce\e-commerce-operations-insights\notebooks


True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime
26517,DEBIT,2,1,36.27,120.89,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Matthew,6833,Watkins,Corporate,PR,4788 Gentle Wynd,725,4,Apparel,18.294920,-66.370522,Cabo Frio,Brasil,6833,2017-04-22 11:05:00,57713,403,9.1,0.07,144374,129.99,0.30,1,129.99,120.89,36.27,South America,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,First Class,04-24-2017,11:05
86957,TRANSFER,2,4,22.17,65.58,Advance shipping,0,29,Shop By Sport,Costa Mesa,EE. UU.,Sharon,2385,Smith,Corporate,CA,1440 Fallen Private,92627,5,Golf,33.635841,-117.904144,Hamm,Alemania,2385,2015-08-13 17:33:00,15396,627,14.4,0.18,38504,39.99,0.34,2,79.98,65.58,22.17,Western Europe,PENDING,627,29,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,08-15-2015,17:33
60693,DEBIT,6,4,-187.19,116.99,Late delivery,1,18,Men's Footwear,Canyon Country,EE. UU.,Ryan,2503,Perkins,Corporate,CA,3240 Foggy Hickory Corner,91351,4,Apparel,33.069271,-116.578644,San Miguelito,PanamÃ¡,2503,2015-05-04 20:19:00,8485,403,13.0,0.10,21185,129.99,-1.60,1,129.99,116.99,-187.19,Central America,ON_HOLD,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,05-10-2015,20:19
120410,TRANSFER,5,4,-36.95,52.79,Late delivery,1,17,Cleats,Orlando,EE. UU.,Helen,6069,Smith,Consumer,FL,2225 Crystal Beach,32825,4,Apparel,28.556259,-81.275871,Turin,Italia,6069,2015-06-30 13:03:00,12369,365,7.2,0.12,30929,59.99,-0.70,1,59.99,52.79,-36.95,Southern Europe,PENDING,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,07-05-2015,13:03
69258,PAYMENT,5,2,-106.46,109.19,Late delivery,1,18,Men's Footwear,Caguas,Puerto Rico,Theresa,9676,Smith,Corporate,PR,4277 Golden Road,725,4,Apparel,18.281818,-66.370583,Watford,Reino Unido,9676,2017-08-05 20:55:00,64934,403,20.8,0.16,162329,129.99,-0.98,1,129.99,109.19,-106.46,Northern Europe,PENDING_PAYMENT,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Second Class,08-10-2017,20:55


In [5]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [6]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [7]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [8]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [9]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [10]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
23304,23305,18.265453,-66.037048,Standard Class
15853,15854,18.205208,-66.370567,Second Class
19965,19966,18.291122,-66.370590,First Class
247,248,33.688961,-117.995743,Standard Class
12610,12611,18.221922,-66.370560,Standard Class


In [11]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [12]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime,store_id
21112,DEBIT,4,4,-41.50,113.09,Shipping on time,0,18,Men's Footwear,Caguas,Puerto Rico,Mary,1853,Austin,Consumer,PR,4090 Colonial End,725,4,Apparel,18.260191,-66.370529,Medan,Indonesia,1853,2015-11-26 08:28:00,22563,403,16.90,0.13,56462,129.99,-0.37,1,129.99,113.09,-41.50,Southeast Asia,COMPLETE,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Standard Class,11-30-2015,08:28,117
99633,PAYMENT,6,4,60.23,165.93,Late delivery,1,46,Indoor/Outdoor Games,Caguas,Puerto Rico,Heather,780,Brennan,Home Office,PR,4899 Rustic Apple Boulevard,725,7,Fan Shop,18.225628,-66.047462,Murfreesboro,Estados Unidos,780,2016-07-30 17:45:00,39510,1014,33.99,0.17,98635,49.98,0.36,4,199.92,165.93,60.23,South of USA,PENDING_PAYMENT,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,08-05-2016,17:45,3657
43748,DEBIT,2,1,114.00,237.50,Late delivery,1,24,Women's Apparel,Elgin,EE. UU.,Patricia,7584,Smith,Consumer,IL,6580 Iron Wagon Inlet,60120,5,Golf,41.976704,-88.260864,AraranguÃ¡,Brasil,7584,2017-06-13 17:40:00,61294,502,12.50,0.05,153335,50.00,0.48,5,250.00,237.50,114.00,South America,COMPLETE,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,First Class,06-15-2017,17:40,9020
125124,PAYMENT,2,2,-61.13,128.69,Shipping on time,0,18,Men's Footwear,Caguas,Puerto Rico,Mary,5504,Brown,Consumer,PR,8495 Lazy Rabbit Diversion,725,4,Apparel,18.235361,-66.370514,Port Harcourt,Nigeria,5504,2016-09-18 12:47:00,42921,403,1.30,0.01,107213,129.99,-0.48,1,129.99,128.69,-61.13,West Africa,PENDING_PAYMENT,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Second Class,09-20-2016,12:47,18590
176494,DEBIT,2,1,35.30,135.75,Late delivery,1,13,Electronics,Caguas,Puerto Rico,Mary,12112,Smith,Consumer,PR,9092 Middle Deer Dale,725,3,Footwear,18.290190,-66.370560,Zamora,EspaÃ±a,12112,2015-07-08 15:30:00,12924,273,4.20,0.03,32322,27.99,0.26,5,139.95,135.75,35.30,Southern Europe,ON_HOLD,273,13,Under Armour Kids' Mercenary Slide,27.99,First Class,07-10-2015,15:30,24157


In [13]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

¡DataFrames del Modelo en Estrella listos en memoria!


In [14]:
USER  = os.getenv("DB_USER")
PASS  = os.getenv("DB_PASSWORD")
HOST  = os.getenv("DB_HOST")
PORT  = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto
DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [15]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
